# Blind Review of the Equipment Rule

`derived/equipment-validation-sample.csv` holds 200 nursing-facility cases, shuffled: 100 that the rule says mention a patient lift, and 100 handling cases that the rule says mention no equipment. The reviewer (the author) read each narrative without seeing the rule's output and marked it `lift`, `aid`, `none`, or `vehicle`. Labels: `derived/equipment-review-labels.csv`. The rule is applied here unchanged from `handling-equipment.ipynb`. Reads only published files.

In [1]:
import re, numpy as np, pandas as pd
s = pd.read_csv('derived/equipment-validation-sample.csv', dtype=str)
l = pd.read_csv('derived/equipment-review-labels.csv', dtype=str)
m = s.merge(l, on='row'); print(len(m), 'cases')

200 cases


In [2]:
EQ_LIFT = (r"\b(?:hoyer|mechanical lift|(?:total|full[- ]body|ceiling|stand(?:ing)?[- ]?up|sit[- ]to[- ]stand)\s*lift|"
           r"sit[- ]to[- ]stand|stand[- ]?aid|stand[- ]?assist(?: device)?|lift sling|lift pad|sling|"
           r"(?:the|a|resident|patient|electric|battery) lift\b(?! (?:with|the resident|resident|patient|him|her))|"
           r"\] lift|lift (?:was|tipped|rolled|bumped|battery|malfunction\w*|failed|broke))")
EQ_AID  = r"\b(?:gait belt|draw ?sheet|slide ?sheet|slide ?board|transfer board|blue pad|bed pad|turn(?:ing)? sheet|lift pad)\b"
EQ_VEH  = r"\b(?:van|bus|vehicle|transport(?:ation)?)\b.{0,60}\blift\b|\blift\b.{0,60}\b(?:van|bus|vehicle)\b|wheelchair lift"

def group(t):
    t = str(t).lower(); v = bool(re.search(EQ_VEH, t)); lf = bool(re.search(EQ_LIFT, t)) and not v
    a = bool(re.search(EQ_AID, t)) and not lf
    return 'lift' if lf else 'aid' if a else 'vehicle' if v else 'none'
m['rule'] = m.narr.map(group)
def wilson(k, n, z=1.96):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return round(p, 3), round(c - h, 3), round(c + h, 3)
ct = pd.crosstab(m.rule, m.reviewer_equipment, rownames=['rule'], colnames=['reviewer'], margins=True)
ct.to_csv('derived/equipment-review-crosstab.csv'); ct

reviewer,aid,lift,none,All
rule,,,,
lift,1,90,9,100
none,1,5,94,100
All,2,95,103,200


## Agreement by rule group, and corrected counts for the 5,654 nursing-facility handling cases

In [3]:
tab = pd.read_csv('derived/equipment-by-group.csv', index_col=0)
H = int(tab['handling cases'].sum()); n_lift = int(tab.loc['patient lift', 'handling cases']); n_none = int(tab.loc['none mentioned', 'handling cases'])
L, N = m[m.rule == 'lift'], m[m.rule == 'none']
pl = wilson(int((L.reviewer_equipment == 'lift').sum()), len(L))
pn = wilson(int((N.reviewer_equipment == 'none').sum()), len(N))
pm = wilson(int((N.reviewer_equipment == 'lift').sum()), len(N))
out = pd.DataFrame([
  {'measure': 'rule says lift: reviewer agrees', 'share': pl[0], 'ci95_low': pl[1], 'ci95_high': pl[2]},
  {'measure': 'rule says none: reviewer agrees', 'share': pn[0], 'ci95_low': pn[1], 'ci95_high': pn[2]},
  {'measure': 'rule says none: reviewer finds a lift', 'share': pm[0], 'ci95_low': pm[1], 'ci95_high': pm[2]}])
est = pd.DataFrame([
  {'quantity': 'handling cases with no equipment mentioned', 'estimate': round(n_none*pn[0]), 'low': round(n_none*pn[1]), 'high': round(n_none*pn[2])},
  {'quantity': 'handling cases with a lift (rule-found, corrected)', 'estimate': round(n_lift*pl[0]), 'low': round(n_lift*pl[1]), 'high': round(n_lift*pl[2])},
  {'quantity': 'handling cases with a lift the rule missed', 'estimate': round(n_none*pm[0]), 'low': round(n_none*pm[1]), 'high': round(n_none*pm[2])}])
est['share of handling cases'] = (est.estimate / H).round(3)
out.to_csv('derived/equipment-review-agreement.csv', index=False); est.to_csv('derived/equipment-corrected-counts.csv', index=False)
print('handling cases:', H); print(out.to_string(index=False)); est

handling cases:

 5654
                              measure  share  ci95_low  ci95_high
      rule says lift: reviewer agrees   0.90     0.826      0.945
      rule says none: reviewer agrees   0.94     0.875      0.972
rule says none: reviewer finds a lift   0.05     0.022      0.112


,quantity,estimate,low,high,share of handling cases
0,handling cases with no equipment mentioned,4520,4208,4674,0.799
1,"handling cases with a lift (rule-found, correc...",554,508,581,0.098
2,handling cases with a lift the rule missed,240,106,539,0.042
